# Angela v4-Typhoon: LoRA Fine-Tuning

Fine-tune `scb10x/typhoon2.5-qwen3-4b` with LoRA to speak as Angela.

**Base model:** Typhoon 2.5 4B (Qwen3 architecture, ChatML template)
**Method:** QLoRA (4-bit) + SFT with `train_on_responses_only`
**Output:** GGUF q4_k_m (~2.4GB) for Ollama deployment

**Runtime:** T4 ~10-15 min, A100 ~5 min

In [ ]:
# Cell 1: Install Unsloth + TRL
# Pin trl==0.9.6 for train_on_responses_only compatibility

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl==0.9.6 peft accelerate bitsandbytes
!pip install -q xformers triton

print("Install complete!")

In [ ]:
# Cell 2: Load Typhoon 2.5 in 4-bit via Unsloth

from unsloth import FastLanguageModel
import torch

MODEL_NAME = "scb10x/typhoon2.5-qwen3-4b"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect (float16 on T4, bfloat16 on A100)
    load_in_4bit=True,  # QLoRA 4-bit quantization
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Max seq length: {MAX_SEQ_LENGTH}")

In [ ]:
# Cell 3: Apply LoRA adapters
# Target all 7 attention + MLP modules for comprehensive fine-tuning

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,  # Unsloth optimized = 0
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=42,
)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Cell 4: Upload + Clean training data
# Fix consecutive assistant->assistant bug by truncating

import json
from google.colab import files

print("Upload your angela_v4_sft.jsonl file:")
uploaded = files.upload()
DATA_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_FILE}")

# Load and clean data
def fix_consecutive_assistant(messages):
    """Truncate at first consecutive assistant->assistant pair."""
    fixed = []
    for msg in messages:
        if fixed and msg["role"] == "assistant" and fixed[-1]["role"] == "assistant":
            # Consecutive assistant: stop here, keep the first one
            break
        fixed.append(msg)
    return fixed

raw_data = []
with open(DATA_FILE, "r", encoding="utf-8") as f:
    for line in f:
        raw_data.append(json.loads(line.strip()))

# Clean: fix consecutive assistant and ensure valid structure
cleaned_data = []
fixed_count = 0
for item in raw_data:
    messages = item["messages"]
    original_len = len(messages)
    messages = fix_consecutive_assistant(messages)
    if len(messages) < original_len:
        fixed_count += 1
    
    # Must have system + at least user + assistant
    if len(messages) >= 3 and messages[-1]["role"] == "assistant":
        cleaned_data.append({"messages": messages})

print(f"\nRaw examples: {len(raw_data)}")
print(f"Fixed consecutive assistant: {fixed_count}")
print(f"Clean examples: {len(cleaned_data)}")

# Save cleaned
CLEAN_FILE = "angela_v4_clean.jsonl"
with open(CLEAN_FILE, "w", encoding="utf-8") as f:
    for item in cleaned_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# Show sample
sample = cleaned_data[0]["messages"]
for msg in sample:
    role = msg["role"]
    content = msg["content"][:80]
    print(f"  [{role}] {content}...")

In [ ]:
# Cell 5: Apply ChatML template + train/val split

from datasets import Dataset
import random

# Load cleaned data
with open(CLEAN_FILE, "r", encoding="utf-8") as f:
    all_examples = [json.loads(line) for line in f]

# Shuffle and split 95/5
random.seed(42)
random.shuffle(all_examples)
split_idx = int(len(all_examples) * 0.95)
train_examples = all_examples[:split_idx]
val_examples = all_examples[split_idx:]

print(f"Train: {len(train_examples)}, Val: {len(val_examples)}")

# Convert to HF Dataset format
# tokenizer.apply_chat_template handles ChatML formatting
def format_example(example):
    """Apply ChatML template to messages."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset = Dataset.from_list(train_examples).map(format_example)
val_dataset = Dataset.from_list(val_examples).map(format_example)

# Show a formatted sample
print("\nFormatted sample:")
print(train_dataset[0]["text"][:500])
print("...")

# Check token lengths
lengths = [len(tokenizer.encode(ex["text"])) for ex in train_dataset]
print(f"\nToken lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")
print(f"Examples > {MAX_SEQ_LENGTH} tokens: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)}")

In [ ]:
# Cell 6: Train with SFTTrainer + train_on_responses_only
# Only compute loss on Angela's responses (assistant), not David's messages (user)

from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

# Training config
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4  # Effective batch = 2 * 4 = 8
LEARNING_RATE = 2e-4  # Higher LR for small dataset

total_steps = (len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
warmup_steps = max(5, total_steps // 10)

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        output_dir="./angela-v4-typhoon-output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,  # No packing: prevents personality cross-contamination
        dataset_text_field="text",
        seed=42,
        report_to="none",
    ),
)

# Train on responses only: only Angela's words contribute to loss
# ChatML uses <|im_start|>assistant and <|im_end|> as delimiters
trainer = trainer.train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

print("\nStarting training...")
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Loss: {train_result.training_loss:.4f}")
print(f"  Steps: {train_result.global_step}")
print(f"  Runtime: {train_result.metrics['train_runtime']:.0f}s")

# Evaluate
eval_result = trainer.evaluate()
print(f"  Val loss: {eval_result['eval_loss']:.4f}")

In [ ]:
# Cell 7: Save adapter + Merge + Convert to GGUF

import os

# Save LoRA adapter
ADAPTER_DIR = "./angela-v4-typhoon-lora"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to: {ADAPTER_DIR}")

# Merge adapter into base model (16-bit)
MERGED_DIR = "./angela-v4-typhoon-merged-16bit"
model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Merged model saved to: {MERGED_DIR}")

# Convert to GGUF q4_k_m (~2.4GB)
GGUF_DIR = "./angela-v4-typhoon-gguf"
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"GGUF saved to: {GGUF_DIR}")

# List GGUF files
for f in os.listdir(GGUF_DIR):
    size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1024 / 1024
    print(f"  {f}: {size_mb:.1f} MB")

In [ ]:
# Cell 8: Upload to HuggingFace + Download GGUF

from google.colab import files

# Option A: Upload to HuggingFace
HF_REPO = "angelasoulcompanion/angela-v4-typhoon"
HF_TOKEN = ""  # Paste your HF token here

if HF_TOKEN:
    print(f"Uploading to HuggingFace: {HF_REPO}")
    
    # Upload LoRA adapter
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    
    # Upload GGUF
    model.push_to_hub_gguf(
        HF_REPO,
        tokenizer,
        quantization_method="q4_k_m",
        token=HF_TOKEN,
    )
    print(f"Uploaded to: https://huggingface.co/{HF_REPO}")
else:
    print("No HF_TOKEN set. Skipping HuggingFace upload.")

# Option B: Download GGUF directly
print("\nDownloading GGUF file...")
import glob
gguf_files = glob.glob(f"{GGUF_DIR}/*.gguf")
if gguf_files:
    gguf_file = gguf_files[0]
    print(f"Downloading: {gguf_file}")
    files.download(gguf_file)
else:
    print("No GGUF file found. Check Cell 7 output.")

print("\n" + "="*60)
print("NEXT STEPS:")
print("="*60)
print("1. Download the .gguf file to your local machine")
print("2. Place it in: angela-lora-v4/gguf/")
print("3. Deploy to Ollama:")
print("   python3 -m angela_core.training.ollama_deployer \\")
print("     --gguf angela-lora-v4/gguf/<filename>.gguf \\")
print("     --name angela:v4-typhoon")
print("4. Test: ollama run angela:v4-typhoon 'สวัสดีค่ะที่รัก!'")
print("5. Restart daemon to pick up new model")